# Locality-Feature Calibration — Phase 0 (diagnostics) + Phase 2 (gate)

Executes `docs/specs/2026-06-27-weighted-adaptive-chunking-plan.md`. The H2 result refuted the cheap **surface** density features. Here we test the **causally-right** *evidence-locality / semantic-segment* features (A1–A5, B1, C4) from `raptor/chunking/segment_features.py` against the per-document optimal leaf size.

**This notebook is cheap & free-tier safe.** It REUSES the cached 50-doc sweep (`granularity_sweep_50.json`) — the only expensive part — and adds a single batched SBERT pass over the 50 docs' sentences (a few minutes on a free T4, seconds if the GPU is warm). **No LLM anywhere.**

**Pre-registered gate (set before looking):** a new feature is ✅ *promising* if held-out `|ρ| ≥ 0.5` **and** its size-map beats fixed in **≥ 30/50** splits; ⚠️ *marginal* at `0.4 ≤ |ρ| < 0.5` (→ scale N / 2-feature combiner); ❌ *dead* if even the label-derived E2/E3 diagnostics don't explain optimal size.


In [ ]:
# 1) Get the code (same repo/branch as the other notebooks).
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

In [ ]:
# 2) Mount Drive and REUSE the cached sweep so nothing expensive reruns.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/raptor_runs'
except Exception as e:
    print('Not on Colab / Drive unavailable -> using local ./raptor_runs', e)
    RUN_DIR = 'raptor_runs'

import os
os.makedirs(RUN_DIR, exist_ok=True)
OUT = os.path.join(RUN_DIR, 'granularity_sweep_50.json')  # the SAME cache the sweep/calibration nbs write
SUMMARY = os.path.join(RUN_DIR, 'locality_calibration_summary.json')
print('RUN_DIR =', RUN_DIR)
print('OUT     =', OUT, '(exists:', os.path.exists(OUT), ')')

# Show the compute device so we know SBERT will use the GPU if present.
try:
    import torch
    print('CUDA available:', torch.cuda.is_available(),
          '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))
except Exception as e:
    print('torch not importable yet:', e)

In [ ]:
# 3) Load the SAME 50 QASPER papers the sweep used (doc_ids must align with the cache).
from experiments.datasets import get_loader

docs = get_loader('qasper').load(limit=50)
docs_by_id = {d.doc_id: d for d in docs}
n_ans = sum(1 for d in docs for q in d.questions if q.evidence)
print(f'{len(docs)} QASPER papers, {n_ans} answerable (gold-evidence) questions total')

In [ ]:
# 4) Load / resume the cached sweep. If the cache is present this is INSTANT
#    (every (doc,size) is skipped); if not, it runs the SBERT-only sweep once.
from experiments import granularity_sweep as gs, granularity_calibration as gc
from collections import Counter

SIZES = [50, 100, 150, 200, 300, 400]
records = gs.run_sweep(docs, SIZES, budget=2000, out_path=OUT)
print(f'{len(records)} (doc, size) records')

h = gc.headroom(records)
print(f"HEADROOM: best fixed = {h['best_fixed_size']} tok @ {h['best_fixed_cov']:.4f} | "
      f"oracle {h['oracle_cov']:.4f} | +{h['rel_gain_pct']:.1f}% rel.")

# The calibration TARGET: per-doc optimal leaf size.
opt = gs.best_size_per_doc(records)
print(f'{len(opt)} docs with a defined optimal size')
print('optimal-size distribution:', dict(sorted(Counter(opt.values()).items())))

In [ ]:
# 5) Compute the NEW locality features for every doc — ONE batched SBERT pass/doc.
#    A4 = mean_segment_len_tokens is the headline candidate.
import time
from tqdm.auto import tqdm
from raptor.EmbeddingModels import SBertEmbeddingModel
from raptor.chunking.segment_features import segment_features, split_sentences

_sbert = SBertEmbeddingModel()._ensure_model()  # SentenceTransformer (uses GPU if available)

def batched_embed(sentences):
    # One encode call for the whole document's sentences = fast + light.
    return _sbert.encode(
        sentences, batch_size=64, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )

feats_new = {}
t0 = time.time()
n_sent_total = 0
for d in tqdm(docs, desc='locality features'):
    n_sent_total += len(split_sentences(d.text))
    feats_new[d.doc_id] = segment_features(d.text, embed_fn=batched_embed)
dt = time.time() - t0
print(f'computed locality features for {len(feats_new)} docs '
      f'({n_sent_total} sentences) in {dt:.1f}s')

# The continuous candidate features we will screen (skip raw counts).
NEW_FEATURES = [
    'mean_segment_len_tokens',  # A4 (headline)
    'paragraph_len_mean',       # B1
    'paragraph_len_cv',         # B1 spread
    'var_adjacent_cosine',      # A2 (regime / lumpiness)
    'topic_shift_rate',         # A3
    'mean_adjacent_cosine',     # A1
    'segment_len_cv',           # A5
    'gzip_ratio',               # C4
]
import pprint; pprint.pprint(feats_new[docs[0].doc_id])

In [ ]:
# ============================ PHASE 0 — DIAGNOSTICS ============================
# E1: per-doc coverage-curve shape. Flat curves = size-insensitive docs = a
#     confound that dilutes any correlation (and the reason Stage-1 gates them).
shapes = gc.coverage_curve_shape(records, flat_eps=0.05)
n_flat = sum(1 for s in shapes.values() if s['is_flat'])
import statistics
flatness_vals = [s['flatness'] for s in shapes.values()]
print(f'E1 curve shape over {len(shapes)} docs:')
print(f"  flat (size-insensitive, flatness<0.05): {n_flat}/{len(shapes)} "
      f"({100*n_flat/max(1,len(shapes)):.0f}%)")
print(f"  flatness mean={statistics.mean(flatness_vals):.3f} "
      f"median={statistics.median(flatness_vals):.3f} max={max(flatness_vals):.3f}")
print('  -> these flat docs will be routed to the fixed default by the Stage-1 regime gate.')

In [ ]:
# E2: gold-evidence span length (the hypothesized LATENT DRIVER of optimal size).
# E3: evidence dispersion (best-effort: locate gold paragraphs in the doc).
import re
from raptor.chunking.segment_features import _PARAGRAPH_SPLIT_RE  # reuse the same splitter

evidence_by_doc = {}
indices_by_doc = {}
matched, total_gold = 0, 0
for d in docs:
    paras = [p.strip() for p in _PARAGRAPH_SPLIT_RE.split(d.text) if p.strip()]
    ev_strings, q_indices = [], []
    for q in d.questions:
        if not q.evidence:
            continue
        ev_strings.extend(q.evidence)
        idxs = []
        for g in q.evidence:
            total_gold += 1
            gg = ' '.join(g.split())
            hit = next((i for i, p in enumerate(paras)
                        if gg and (gg in ' '.join(p.split()))), None)
            if hit is not None:
                matched += 1
                idxs.append(hit)
        if idxs:
            q_indices.append(idxs)
    evidence_by_doc[d.doc_id] = ev_strings
    indices_by_doc[d.doc_id] = q_indices

e2 = gc.gold_evidence_span_lengths(evidence_by_doc)   # {doc: mean tokens / gold para}
e3 = gc.evidence_dispersion(indices_by_doc)           # {doc: mean span of gold idxs}
print(f'E3 gold-paragraph match rate: {matched}/{total_gold} '
      f'({100*matched/max(1,total_gold):.0f}%) — dispersion is best-effort.')

# Do the label-derived diagnostics explain optimal size? (Spearman over shared docs.)
ids = [k for k in opt if k in e2]
print(f"\nE2 gold-span-length vs optimal size : rho = "
      f"{gc.rank_corr([e2[k] for k in ids], [opt[k] for k in ids]):+.3f}  (n={len(ids)})")
ids3 = [k for k in opt if k in e3 and e3[k] > 0]
if len(ids3) >= 2:
    print(f"E3 dispersion       vs optimal size : rho = "
          f"{gc.rank_corr([e3[k] for k in ids3], [opt[k] for k in ids3]):+.3f}  (n={len(ids3)})")
print('\nREAD: if BOTH E2 and E3 are near-zero, the per-doc target is irreducibly '
      'noisy at this N -> the negative-result path is the honest outcome (Phase 6).')

In [ ]:
# ============================ PHASE 2 — SINGLE-FEATURE GATE ====================
# (a) Spearman of each NEW feature vs optimal size, AND vs E2 (the latent driver).
ids = [k for k in opt if k in feats_new]
opt_vec = [opt[k] for k in ids]
e2_vec = [e2[k] for k in ids]

print(f"{'feature':<26}{'rho vs OPT size':>16}{'rho vs E2':>14}")
print('-' * 56)
rows = []
for f in NEW_FEATURES:
    xs = [feats_new[k][f] for k in ids]
    r_opt = gc.rank_corr(xs, opt_vec)
    r_e2 = gc.rank_corr(xs, e2_vec)
    rows.append((f, r_opt, r_e2))
for f, r_opt, r_e2 in sorted(rows, key=lambda t: abs(t[1]), reverse=True):
    flag = '  <== |rho|>=0.5' if abs(r_opt) >= 0.5 else ('  ~marginal' if abs(r_opt) >= 0.4 else '')
    print(f'{f:<26}{r_opt:>16.3f}{r_e2:>14.3f}{flag}')

# Reference: the OLD surface best was type_token_ratio at -0.379. Beat it.
print('\n(reference: old surface best |rho| was 0.379 — the bar for "interesting" is ~0.5)')

In [ ]:
# (b) Held-out 50-split gate: does a NEW-feature size-map beat the tuned fixed size?
#     Reuses the exact H2 protocol (select_and_fit on TRAIN, score on TEST).
import statistics
from tqdm.auto import tqdm

N_SPLITS = 50
bf_size, _ = gc.best_global_fixed(records)
_, oracle_sizes_all = gc.oracle_per_doc(records)

rows = {'best global fixed': [], 'NEW selected feature': [], 'per-doc oracle': []}
wins_new = 0
sel_freq = Counter()

for seed in tqdm(range(N_SPLITS), desc='held-out splits'):
    tr, te = gc.train_test_split_docs(list(opt), seed=seed)
    if not tr or not te:
        continue
    cov_fixed = gc.coverage_under_sizes(records, {k: bf_size for k in te})
    sel = gc.select_and_fit(feats_new, opt, tr, feature_names=NEW_FEATURES)
    sel_freq[sel['feature']] += 1
    pred = ({k: gc.predict_size(feats_new[k][sel['feature']], sel['fit'], 50, 400)
             for k in te} if sel['feature'] else {})
    cov_new = gc.coverage_under_sizes(records, pred)
    cov_oracle = gc.coverage_under_sizes(records, {k: oracle_sizes_all[k] for k in te
                                                   if k in oracle_sizes_all})
    rows['best global fixed'].append(cov_fixed)
    rows['NEW selected feature'].append(cov_new)
    rows['per-doc oracle'].append(cov_oracle)
    if cov_new > cov_fixed:
        wins_new += 1

n = len(rows['best global fixed'])
print(f"\n{'policy (mean TEST coverage over %d splits)'%n:<34}{'mean':>9}{'std':>9}")
print('-' * 52)
for k, v in rows.items():
    print(f'{k:<34}{statistics.mean(v):>9.4f}{statistics.pstdev(v):>9.4f}')
print(f"\nNEW-feature beats fixed in {wins_new}/{n} splits ({100*wins_new/max(1,n):.0f}%)")
print('selected-feature frequency:', dict(sel_freq.most_common()))

# ---- Verdict against the pre-registered gate ----
best_feat, best_rho = max(((f, gc.rank_corr([feats_new[k][f] for k in ids], opt_vec))
                           for f in NEW_FEATURES), key=lambda t: abs(t[1]))
winrate = wins_new / max(1, n)
if abs(best_rho) >= 0.5 and wins_new >= 30:
    verdict = 'PROMISING -> build the Phase-3 weighted combiner & confirm end-to-end'
elif abs(best_rho) >= 0.4:
    verdict = 'MARGINAL -> scale to ~150 docs and/or try a 2-feature combiner'
else:
    verdict = 'DEAD on the cheap route -> strengthen the NEGATIVE result (Phase 6)'
print(f"\nVERDICT: best new feature = {best_feat} (|rho|={abs(best_rho):.3f}), "
      f"win-rate={winrate:.0%}  ==>  {verdict}")

In [ ]:
# ============ PHASE 2 (C) — STRATIFIED HEDGE: size-sensitive docs only ========
# E1 showed ~22% of docs are size-INSENSITIVE (flat oracle curve); they inject a
# confound. Restrict to docs where size demonstrably matters and ask: does ANY
# signal appear? If it is dead even here, the negative result is bulletproof.
sensitive = [k for k in opt if k in shapes and not shapes[k]['is_flat']]
print(f'size-sensitive docs (non-flat): {len(sensitive)}/{len(opt)}')

sids = [k for k in sensitive if k in feats_new]
sopt = [opt[k] for k in sids]
se2 = [e2[k] for k in sids]
print(f"\nE2 gold-span-length vs optimal size (sensitive only): "
      f"rho = {gc.rank_corr(se2, sopt):+.3f}  (n={len(sids)})")
print(f"\n{'feature':<26}{'rho vs OPT (sensitive)':>24}")
print('-' * 50)
strat_rho = {}
for f in NEW_FEATURES:
    r = gc.rank_corr([feats_new[k][f] for k in sids], sopt)
    strat_rho[f] = r
for f, r in sorted(strat_rho.items(), key=lambda t: abs(t[1]), reverse=True):
    flag = '  <== |rho|>=0.5' if abs(r) >= 0.5 else ('  ~marginal' if abs(r) >= 0.4 else '')
    print(f'{f:<26}{r:>24.3f}{flag}')

# Held-out gate on the sensitive subset (fixed baseline also restricted to it).
srows = {'fixed': [], 'NEW selected': [], 'oracle': []}
swins = 0
for seed in tqdm(range(N_SPLITS), desc='sensitive splits'):
    tr, te = gc.train_test_split_docs(sensitive, seed=seed)
    if not tr or not te:
        continue
    bf = gc.coverage_under_sizes(records, {k: bf_size for k in te})
    sel = gc.select_and_fit(feats_new, opt, tr, feature_names=NEW_FEATURES)
    pred = ({k: gc.predict_size(feats_new[k][sel['feature']], sel['fit'], 50, 400)
             for k in te} if sel['feature'] else {})
    cn = gc.coverage_under_sizes(records, pred)
    co = gc.coverage_under_sizes(records, {k: oracle_sizes_all[k] for k in te
                                           if k in oracle_sizes_all})
    srows['fixed'].append(bf); srows['NEW selected'].append(cn); srows['oracle'].append(co)
    if cn > bf:
        swins += 1
sn = len(srows['fixed'])
print(f"\n{'policy (sensitive-only TEST)':<22}{'mean':>9}{'std':>9}")
print('-' * 40)
for k, v in srows.items():
    print(f'{k:<22}{statistics.mean(v):>9.4f}{statistics.pstdev(v):>9.4f}')
strat_best = max(strat_rho.items(), key=lambda t: abs(t[1]))
print(f"\nSENSITIVE-ONLY: new-feature beats fixed {swins}/{sn} ({100*swins/max(1,sn):.0f}%); "
      f"best feature {strat_best[0]} |rho|={abs(strat_best[1]):.3f}")
print('READ: if this is ALSO dead, the negative result holds even where size matters most.')

strat = {
    'n_sensitive': len(sensitive),
    'e2_rho_sensitive': gc.rank_corr(se2, sopt),
    'feature_rho_vs_opt_sensitive': strat_rho,
    'held_out_sensitive': {k: {'mean': statistics.mean(v), 'std': statistics.pstdev(v)}
                           for k, v in srows.items()},
    'new_feature_winrate_sensitive': swins / max(1, sn),
}

In [ ]:
# 6) Persist a compact summary to Drive for the result writeup.
import json
summary = {
    'stratified_sensitive': strat,
    'n_docs': len(opt),
    'headroom': h,
    'optimal_size_distribution': dict(sorted(Counter(opt.values()).items())),
    'e1_flat_fraction': n_flat / max(1, len(shapes)),
    'e2_rho_span_len_vs_opt': gc.rank_corr([e2[k] for k in ids], opt_vec),
    'feature_rho_vs_opt': {f: gc.rank_corr([feats_new[k][f] for k in ids], opt_vec)
                           for f in NEW_FEATURES},
    'feature_rho_vs_e2': {f: gc.rank_corr([feats_new[k][f] for k in ids], e2_vec)
                          for f in NEW_FEATURES},
    'held_out': {k: {'mean': statistics.mean(v), 'std': statistics.pstdev(v)}
                 for k, v in rows.items()},
    'new_feature_winrate': winrate,
    'selected_feature_freq': dict(sel_freq),
    'verdict': verdict,
}
with open(SUMMARY, 'w') as fh:
    json.dump(summary, fh, indent=2, default=float)
print('wrote', SUMMARY)
print(json.dumps(summary, indent=2, default=float))

## How to read this

1. **E1 (flat fraction).** If many docs are flat (size-insensitive), correlations are diluted — the Stage-1 regime gate exists precisely to route those to the fixed default.
2. **E2 / E3.** These are *label-derived* (near-oracle) signals. If even they are near-zero against optimal size, no cheap document feature can win — that is the honest stop signal.
3. **Phase-2 single-feature gate.** Compare each new feature's `|ρ|` to the **old surface best (0.379)** and the **bar (~0.5)**. `rho vs E2` says whether the feature tracks the latent driver.
4. **Held-out 50-split.** The headline statistic: does the new-feature size-map beat the tuned fixed size, and how often? Report **mean ± std** and **win-rate**.
5. **Verdict.** Drives the next phase per the pre-registered decision tree (promising → Phase 3 combiner; marginal → scale N; dead → strengthen the negative result).

Copy the printed summary (and `locality_calibration_summary.json` on Drive) into a `docs/results/2026-06-27-locality-calibration.md` writeup.
